In [2]:
import numpy as np                        # For linear algebra
import cvxpy as cp
import matplotlib.pyplot as plt            # For plots
from scipy.signal import ss2tf
# np.random.seed(1)                          # Generate random seed
# np.set_printoptions(precision=1)           # Set nice printing format

In [3]:
#1. Generate random system

def is_controllable(A, B, tol=1e-9):
    """
    Check controllability of (A,B) using the controllability matrix:
        C = [B, AB, A^2 B, ..., A^{n-1} B]
    """
    n = A.shape[0]
    C = B
    for _ in range(1, n):
        C = np.hstack((C, A @ C[:, -B.shape[1]:]))
    rank = np.linalg.matrix_rank(C, tol=tol)
    return rank == n

def is_observable(A, C, tol=1e-9):
    """
    Check observability of (A,C) using the observability matrix:
        O = [C; CA; CA^2; ...; CA^{n-1}]
    """
    n = A.shape[0]
    O = C
    for _ in range(1, n):
        O = np.vstack((O, O[-C.shape[0]:, :] @ A))
    rank = np.linalg.matrix_rank(O, tol=tol)
    return rank == n

def random_discrete_system(n=2, m=1, p=1, max_tries=1000, scale_A=1.0):
    """
    Generate a random discrete-time state-space system:
        x_{k+1} = A x_k + B u_k
        y_k     = C x_k + D u_k

    such that (A,B) is controllable and (A,C) is observable.

    n : number of states
    m : number of inputs
    p : number of outputs
    max_tries : how many random draws before giving up
    scale_A : multiply A by this factor (you can <1.0 to make it "more stable"
    """
    for _ in range(max_tries):
        A = scale_A * np.random.randn(n, n)
        B = np.random.randn(n, m)
        C = np.random.randn(p, n)
        D = np.zeros((p, m))  # often we take D = 0

        if is_controllable(A, B) and is_observable(A, C):
            return A, B, C, D

    raise RuntimeError("Failed to find controllable & observable system in max_tries.")

def ss_to_tf_discrete(A, B, C, D):
    """
    Compute discrete-time transfer function G(z) from (A,B,C,D).

    Returns:
        num: array of numerator coefficient arrays, shape (p, m), each entry is 1D ndarray
        den: 1D ndarray of denominator coefficients (same for all I/O pairs)
    """
    p, m = D.shape
    num = np.empty((p, m), dtype=object)
    den = None

    for j in range(m):
        # ss2tf returns numerator(s) for all outputs for input j
        num_j, den_j = ss2tf(A, B, C, D, input=j)
        if den is None:
            den = den_j
        for i in range(p):
            num[i, j] = num_j[i, :]

    return num, den

In [ ]:
#2 Generate two systems
# Generate random system
A10, B10, C10, D10 = random_discrete_system(n=2, m=1, p=1, scale_A=0.5)
A20, B20, C20, D20 = random_discrete_system(n=2, m=1, p=1, scale_A=0.5)
print("A10 =\n", A10)
print("A20 =\n", A20)
print("B10 =\n", B10)
print("B20 =\n", B20)
print("C10 =\n", C10)
print("C20 =\n", C20)
print("D10 =\n", D10)
print("D20 =\n", D20)

# #An exmaple
# A10 =
#  [[0.19615359 0.50849891]
#  [0.46650182 0.17302113]]
# A20 =
#  [[-0.11976355  0.03018679]
#  [-0.91423518  0.70391251]]
# B10 =
#  [[-0.40165864]
#  [-0.69455835]]
# B20 =
#  [[-1.81248007]
#  [-0.51106187]]

A10 =
 [[-0.10249883 -1.2293472 ]
 [-0.86806542  0.21183228]]
A20 =
 [[-0.94388088  0.27524258]
 [-1.16074585 -0.36489908]]
B10 =
 [[-0.42766911]
 [ 0.29744984]]
B20 =
 [[0.25226906]
 [2.68165053]]
C10 =
 [[-0.90939025  1.37173745]]
C20 =
 [[0.26501788 0.9008304 ]]
D10 =
 [[0.]]
D20 =
 [[0.]]


In [14]:
#2.2 Assign and simulate two systems
A1, B1, C1, D1 = A10, B10, C10, D10
A2, B2, C2, D2 = A20, B20, C20, D20

In [16]:
#3.1 Find transfer function of both systems
num1, den1 = ss_to_tf_discrete(A1, B1, C1, D1)
num2, den2 = ss_to_tf_discrete(A2, B2, C2, D2)
print("num1 =", num1)
print("den1 =", den1)
print("num2 =", num2)
print("den2 =", den2)
# z_num1 = np.poly1d(num1[0,0])
# z_den1 = np.poly1d(den1)
# print("\nG1(s) = (", z_num1, " ) / (", z_den1, ")")
# z_num2 = np.poly1d(num2[0,0])
# z_den2 = np.poly1d(den2)
# print("\nG2(s) = (", z_num2, " ) / (", z_den2, ")")

def eval_tf(num, den, z):
    """
    Evaluate a MIMO transfer function matrix P(z) at a complex point z.

    num: array of shape (p, m), each entry is a 1D array of numerator coeffs
    den: 1D array of denominator coeffs
    z: complex scalar (e^{jω} in discrete-time)

    Returns: P(z) as a (p, m) complex ndarray.
    """
    p, m = num.shape
    P = np.zeros((p, m), dtype=complex)
    den_val = np.polyval(den, z)
    for i in range(p):
        for j in range(m):
            P[i, j] = np.polyval(num[i, j], z) / den_val
    return P

P1_z = eval_tf(num1, den1, z=np.exp(1j * 0))
P2_z = eval_tf(num2, den2, z=np.exp(1j * 0))
print("\nP1(e^{j0}) =\n", P1_z)
print("\nP2(e^{j0}) =\n", P2_z)

num1 = [[array([ 0.        , -1.73246924,  0.00816364])]]
den1 = [1.         0.49524186 0.08148176]
num2 = [[array([ 0.        , -3.03647515,  0.57236926])]]
den2 = [ 1.         -0.47870095  0.00344089]

P1(e^{j0}) =
 [[-1.09360041+0.j]]

P2(e^{j0}) =
 [[-4.69586111+0.j]]


In [18]:
#4 Steps to calculate wno
def inv_sqrt_hermitian(M, eps=1e-12):
    """
    Compute (M)^{-1/2} for a Hermitian positive-definite matrix M.
    """
    # Hermitian eigen-decomposition
    lam, V = np.linalg.eigh(M)
    lam_clipped = np.clip(lam, eps, None)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(lam_clipped))
    return V @ D_inv_sqrt @ V.conj().T

def L_mat(P1,P2):
    p, m = P1.shape
    I_p = np.eye(p, dtype=complex)
    return inv_sqrt_hermitian(I_p + P1 @ P2.conj().T)

def R_mat(P1,P2):
    p, m = P1.shape
    I_m = np.eye(m, dtype=complex)
    return inv_sqrt_hermitian(I_m + P1.conj().T @ P2)

def G_star_G(P1, P2):
    """
    Compute G1^* G2 at a single frequency, given P1(z), P2(z).
    """
    p, m = P1.shape
    R1 = R_mat(P1, P1)
    R2 = R_mat(P2, P2)
    middle = np.eye(m, dtype=complex) + P2.conj().T @ P1
    return R1.conj().T @ middle @ R2

def winding_number(det_vals):
    """
    Approximate winding number of a complex function around 0,
    given samples det_vals along a closed contour (here unit circle).
    """
    # Argument at each sample
    angles = np.unwrap(np.angle(det_vals))
    # Net change in angle over the loop
    dtheta = angles[-1] - angles[0]
    return int(np.round(dtheta / (2 * np.pi)))


In [19]:
def v_gap_discrete(num1, den1, num2, den2,
                   n_freq=2000, eps_det=1e-6):
    """
    Approximate Vinnicombe v-gap δ_v(P1, P2) for discrete-time plants
    given by transfer-function data (num, den).

    num1, num2: (p, m) arrays of 1D numerator coefficient arrays
    den1, den2: 1D arrays of denominator coefficients
    n_freq: number of frequency samples on unit circle
    eps_det: tolerance for det(G1^* G2) being nonzero

    Returns:
        delta_v: the v-gap in [0, 1]
        v_r:     the 'v_r' value ||G2 G1||_∞ before the winding-number test
    """
    # Frequency grid on unit circle (closed loop from -π to π)
    omegas = np.linspace(-np.pi, np.pi, n_freq)
    v_r = 0.0

    det_vals = []

    for w in omegas:
        z = np.exp(1j * w)

        # Evaluate transfer matrices at this frequency
        P1 = eval_tf(num1, den1, z)
        P2 = eval_tf(num2, den2, z)

        # --- a_v part: || L(P2) (P2 - P1) R(P1) ||_2 ---
        L2 = L_mat(P2,P2)
        R1 = R_mat(P1,P1)
        M = L2 @ (P2 - P1) @ R1   # p x m
        # spectral norm (largest singular value)
        v_r = max(v_r, np.linalg.norm(M, 2))

        # --- winding-number part: det(G1^* G2) ---
        G1G2_star = G_star_G(P1, P2)
        det_val = np.linalg.det(G1G2_star)
        det_vals.append(det_val)

    det_vals = np.array(det_vals)

    # Check det != 0 everywhere on the grid
    if np.min(np.abs(det_vals)) < eps_det:
        return 1.0, v_r

    # Approximate winding number of det(G1^* G2)
    wno = winding_number(det_vals)

    if wno != 0:
        return 1.0, v_r

    return float(v_r), v_r

delta_v, v_r = v_gap_discrete(num1, den1, num2, den2)
print("v-gap δ_v(P1,P2) ≈", delta_v)
print("v_r(P1,P2) ≈", v_r)


v-gap δ_v(P1,P2) ≈ 0.5063101540448839
v_r(P1,P2) ≈ 0.5063101540448839


In [ ]:
#TODO find v gap boundary